# Politieke Reizen & Geschenken

**Thema:** Buitenlandse reizen en ontvangen geschenken door politici.

**Kernvragen:** Naar welke bestemmingen reizen Kamerleden het vaakst, wat is het hoofddoel, en wie betaalt deze reizen? Welke trends zijn er te zien in ontvangen geschenken?

Dit notebook verkent o.a. de silver-tabellen `persoon`, `persoon_reis` en `persoon_geschenk`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

## Naar welke bestemmingen reizen Kamerleden het vaakst?

In [ ]:
aantal_keer_bestemming = query_df("""
  SELECT bestemming, COUNT(id) as aantal_reizen
  FROM silver.persoon_reis
  GROUP BY bestemming
  ORDER BY aantal_reizen DESC
""")
 
aantal_keer_bestemming

De 3 meest voorkomende bestemmingen zijn Brussel, Berlijn en Parijs.

## Wat voor doelen komen het vaakst voor?

In [ ]:
reisdoelen = query_df("""
  SELECT doel, COUNT(doel) as aantal
  FROM silver.persoon_reis
  GROUP BY doel
  ORDER BY aantal DESC
""")
 
display(reisdoelen)
for i in range(5):
    print(reisdoelen['doel'].iloc[i])

Vooral werkbezoeken komen het vaakst voor.

## Hoelang zijn de reizen?

Het aantal dagen dat een gemaakte reis duurt.

In [ ]:
# Schrijf hier je Python/SQL code
duur_dagen = query_df("""
  SELECT id, sum(tot_en_met - van) as duur_dagen
  FROM silver.persoon_reis
  GROUP BY id
  ORDER BY duur_dagen DESC
""")
 
duur_dagen

In [ ]:
data = duur_dagen["duur_dagen"]
plt.hist(data, bins=np.arange(data.min(), data.max()+2))
plt.show()

In [ ]:
sum(data.isna())

Er zijn in totaal 30 missende waarden

In [ ]:
data = duur_dagen["duur_dagen"]

len_before = len(data)

Q1 = data.quantile(0.25)
Q3 = data.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

upper_array = np.where(data >= upper)[0]
lower_array = np.where(data <= lower)[0]

data.drop(index=upper_array, inplace=True)
data.drop(index=lower_array, inplace=True)

plt.hist(data, bins=np.arange(data.min(), data.max()+2), align = "left")
plt.show()

len_after = len(data)

print("Old length: {}".format(len_before))
print("New length: {}".format(len_after))
print("# Outliers removed: {}".format(len_before - len_after))

Om beter beeld te krijgen hoe de reisduur eruit ziet, moesten de uitbijters verwijderd worden. Hier zie je dat deze meeste reizen 1 of 2 dagen duren.

## Wanneer zijn de reizen?

De datums van de reizen.

In [ ]:
reis_datums = query_df("""
    SELECT van, tot_en_met
    FROM silver.persoon_reis
    ORDER BY van
""")

print(len(reis_datums))

for col in reis_datums.columns:
    print(sum(reis_datums[col].isna()))
reis_datums = reis_datums.dropna()

print(len(reis_datums))
print(str(round(30/3339 * 100, 2)) + '% zijn missende waarden')
reis_datums

Er is besloten om alle missende waarden te verwijderen, er zijn in totaal 29 rijen volledig missende waarden en 1 rij waar alleen de einddatum mist. Dit is 0.9% van de reizen. Er zijn in totaal 3309 reizen geweest waarvoor alles is genoteerd. Hieruit kan gekeken worden wanneer de reizen zijn geweest en in welke periode het meest wordt gereisd.

## Wie betaalt de reizen?

In [ ]:
door_wie_betaald = query_df("""
    SELECT betaald_door, COUNT(id) as aantal
    FROM silver.persoon_reis
    GROUP BY betaald_door
    ORDER BY aantal DESC
""")

door_wie_betaald

De meeste reizen worden betaald door de Tweede Kamer der Staten-Generaal.

## Welke partij (fractielabel) ontvangt de meeste geschenken?

Elke rij een fractielabel en het aantal geschenken die de partij heeft ontvangen.

In [ ]:
partijen_aantal_geschenken = query_df("""
    SELECT p.fractielabel, COUNT(g.omschrijving) as aantal_geschenken
    FROM silver.persoon as p
    INNER JOIN silver.persoon_geschenk as g
    ON g.persoon_id = p.id
    GROUP BY p.fractielabel
    ORDER BY aantal_geschenken DESC
""")

partijen_aantal_geschenken

Er zijn best veel missende waarden, er is besloten om te kijken hoeveel personen een fractielabel hebben.

In [ ]:
aantal_personen_per_fractie = query_df("""
    SELECT fractielabel, COUNT(id) as aantal_personen
    FROM silver.persoon
    GROUP BY fractielabel
    ORDER BY aantal_personen DESC
""")
display(aantal_personen_per_fractie)
print(str(round(100 - 2977 / sum(aantal_personen_per_fractie['aantal_personen']) * 100, 2)) + "% van de personen hebben een fractielabel")

Van de meeste geschenken is het onbekend welke partij het geschenk heeft ontvangen. Dit kan komen door het nooit invullen van het fractielabel of te komen overlijden waardoor de fractielabel waarschijnlijk is verwijderd. In beide gevallen zouden hun geschenken onder NaN vallen. Het is dus niet mogelijk om 100% zeker te zijn welke partij de meeste geschenken heeft ontvangen. Als wij puur kijken naar de personen die wel een fractielabel hebben, heeft de CDA de meeste geschenken ontvangen.

In [ ]:
aantal_reizen_per_fractie = query_df("""
    SELECT fractielabel, count(r.id) as aantal_reizen
    FROM silver.persoon_reis as r
    RIGHT JOIN silver.persoon as p
    ON p.id = r.persoon_id
    GROUP BY fractielabel
    ORDER BY aantal_reizen DESC
""")

display(aantal_reizen_per_fractie)

Omdat er zo weinig personen zijn die een fractielabel hebben, is het moeilijk 100% te weten welke partij het meest reist. Als er puur wordt gekeken naar de personen die wel een fractielabel hebben, heeft de CDA het meest gereisd.

## Welk kamerlid gaat het vaakst op reis?

De persoon die het meest reist.

In [ ]:
aantal_reizen_per_persoon = query_df("""
    SELECT roepnaam, achternaam, count(r.id) as aantal_reizen, overlijdensdatum
    FROM silver.persoon_reis as r
    RIGHT JOIN silver.persoon as p
    ON p.id = r.persoon_id
    WHERE functie != 'Oud Kamerlid'
    GROUP BY roepnaam, achternaam, overlijdensdatum
    ORDER BY aantal_reizen DESC
""")

null_data = aantal_reizen_per_persoon[aantal_reizen_per_persoon['overlijdensdatum'].isnull()]

# Checken of de code werkt en alle levende personen behoudt, als 0 -> de code werkt.
val = sum(null_data['overlijdensdatum'].isna()) - len(null_data)
if val != 0:
    print("Iets ging mis.")
else:
    roepnaam = null_data.iloc[0]['roepnaam']
    achternaam = null_data.iloc[0]['achternaam']
    aantal_reizen = null_data.iloc[0]['aantal_reizen']
    print("Het kamerlid wie het meest heeft gereisd is: {} {}. {} keer gereisd.".format(roepnaam, achternaam, aantal_reizen))

## Welke geschenken komen van politieke groepen?

De geschenken gegeven door polikieke partijen.

In [ ]:
query_tekst = """
    SELECT omschrijving
    FROM silver.persoon_geschenk
    WHERE 1 = 1
    AND (omschrijving LIKE '%CDA%'
    """

lijst_partijen = list(aantal_reizen_per_fractie['fractielabel'].dropna())
print(lijst_partijen)

lijst_partijen.pop(0) 

print(lijst_partijen)

for partij in lijst_partijen:
    query_text += "\n\tOR omschrijving LIKE '%{}%'".format(partij)

query_tekst += ")"
print(query_text)

geschenken_van_partijen = query_df(query_text)
display(geschenken_van_partijen)
print(geschenken_van_partijen.iloc[271]['omschrijving'])
# Voorbeeld in geval van df verandering:
# Ontvangen van Marianne Thieme het boek 'Het gelijk van de dieren, het geluk van de mensen' ter waarde van €14,95. Geplaatst in bibliotheek Werkgroep Buitenland SP.

Het is erg moeilijk om te weten of de partij het geschenk heeft ontvangen of het geschenk heeft gegeven. Het zou mogelijk kunnen zijn als alle omschrijvingen beginnen met "Ontvangen van" of "Aangeboden door". Het beste zou zijn geweest als er nog 2 kolommen waren die aangeven wie de ontvanger is en wie de gever is.